# LLM Preference Prediction with DistilBERT (Kaggle-ready)

This notebook trains a DistilBERT classifier for the Chatbot Arena preference task and creates `submission.csv`.

Pipeline:
- Robust parsing of list-formatted fields (`prompt`, `response_a`, `response_b`)
- EDA + sequence-length analysis
- DistilBERT fine-tuning with strong Kaggle settings
- Response-swap TTA at inference to reduce position bias
- Competition-format submission file


In [ ]:
import ast
import json
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

from sklearn.metrics import accuracy_score, log_loss
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.utils.class_weight import compute_class_weight

from datasets import Dataset
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    EarlyStoppingCallback,
    Trainer,
    TrainingArguments,
)

warnings.filterwarnings('ignore')

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(42)
print('CUDA available:', torch.cuda.is_available())


CUDA available: False


In [ ]:
is_kaggle = Path('/kaggle').exists()
print(f'Running on Kaggle: {is_kaggle}')

if is_kaggle:
    # Check if model dataset is available
    model_input_path = Path('/kaggle/input/datasets/jozefmakis/my-distilbert')
    if not model_input_path.exists():
        print('\n⚠️  WARNING: distilbert-base-uncased model input not found!')
        print('The notebook will try to download from HuggingFace, but this may fail without internet.')
        print('\nTo fix on Kaggle: click "+ Add input" → search "distilbert-base-uncased" → attach it')


In [ ]:
KAGGLE_DATA_DIR = Path('/kaggle/input/competitions/llm-classification-finetuning')
LOCAL_DATA_DIR = Path('./data')

DATA_DIR = KAGGLE_DATA_DIR if KAGGLE_DATA_DIR.exists() else LOCAL_DATA_DIR
print('Using DATA_DIR:', DATA_DIR)

train_df = pd.read_csv(DATA_DIR / 'train.csv')
test_df = pd.read_csv(DATA_DIR / 'test.csv')
sample_sub = pd.read_csv(DATA_DIR / 'sample_submission.csv')

print('train shape:', train_df.shape)
print('test shape :', test_df.shape)
train_df.head(2)


Using DATA_DIR: data


train shape: (57477, 9)
test shape : (3, 4)


,id,model_a,model_b,prompt,response_a,response_b,winner_model_a,winner_model_b,winner_tie
0,30192,gpt-4-1106-preview,gpt-4-0613,"[""Is it morally right to try to have a certain...","[""The question of whether it is morally right ...","[""As an AI, I don't have personal beliefs or o...",1,0,0
1,53567,koala-13b,gpt-4-0613,"[""What is the difference between marriage lice...","[""A marriage license is a legal document that ...","[""A marriage license and a marriage certificat...",0,1,0


In [ ]:
LABEL_COLS = ['winner_model_a', 'winner_model_b', 'winner_tie']

def parse_turns(x):
    if pd.isna(x):
        return []
    if isinstance(x, list):
        return [str(t) for t in x]

    s = str(x).strip()
    if not s:
        return []

    for parser in (ast.literal_eval, json.loads):
        try:
            val = parser(s)
            if isinstance(val, list):
                return [str(t) for t in val]
            return [str(val)]
        except Exception:
            pass

    return [s]

def sanitize_text(s):
    s = str(s)
    return s.encode('utf-8', errors='replace').decode('utf-8')

def normalize_text(s):
    return ' '.join(sanitize_text(s).replace('\n', ' ').split())

def clip_head_tail_words(text, max_words):
    words = str(text).split()
    if len(words) <= max_words:
        return ' '.join(words)
    head = int(max_words * 0.7)
    tail = max_words - head
    return ' '.join(words[:head] + ['[SNIP]'] + words[-tail:])

def join_turns(x):
    turns = [normalize_text(t) for t in parse_turns(x)]
    turns = [t for t in turns if t]
    return ' [TURN] '.join(turns)

def build_input(prompt, response_a, response_b):
    p = clip_head_tail_words(join_turns(prompt), max_words=90)
    a = clip_head_tail_words(join_turns(response_a), max_words=150)
    b = clip_head_tail_words(join_turns(response_b), max_words=150)
    return sanitize_text(f'[PROMPT] {p} [SEP] [RESPONSE_A] {a} [SEP] [RESPONSE_B] {b}')

train_df['text'] = train_df.apply(
    lambda r: build_input(r['prompt'], r['response_a'], r['response_b']), axis=1
)
train_df['text_swapped'] = train_df.apply(
    lambda r: build_input(r['prompt'], r['response_b'], r['response_a']), axis=1
)
test_df['text'] = test_df.apply(
    lambda r: build_input(r['prompt'], r['response_a'], r['response_b']), axis=1
)
test_df['text_swapped'] = test_df.apply(
    lambda r: build_input(r['prompt'], r['response_b'], r['response_a']), axis=1
)

train_df['label'] = train_df[LABEL_COLS].values.argmax(axis=1).astype(int)

print('Class distribution:')
print(train_df['label'].value_counts(normalize=True).sort_index())
train_df[['id', 'text', 'label']].head(2)

Class distribution:
label
0    0.349079
1    0.341911
2    0.309011
Name: proportion, dtype: float64


,id,text,label
0,30192,[PROMPT] Is it morally right to try to have a ...,0
1,53567,[PROMPT] What is the difference between marria...,1


In [ ]:
train_df['char_len'] = train_df['text'].str.len()
train_df['word_len'] = train_df['text'].str.split().str.len()

print(train_df[['char_len', 'word_len']].describe(percentiles=[0.5, 0.9, 0.95, 0.99]))

p95_words = int(train_df['word_len'].quantile(0.95))
MAX_LENGTH = int(np.clip(p95_words * 1.6, 192, 512))
print(f'MAX_LENGTH selected: {MAX_LENGTH} (based on p95 words={p95_words})')


In [ ]:
MODEL_CANDIDATES = [
    '/kaggle/input/models/shepscientific/distilbert-base-uncased/transformers/distilbert-base-uncased-v1/1',
    '/kaggle/input/datasets/jozefmakis/my-distilbert',
    'distilbert-base-uncased',
]
MODEL_NAME = next((m for m in MODEL_CANDIDATES if Path(m).exists()), MODEL_CANDIDATES[-1])
print('Using MODEL_NAME:', MODEL_NAME)

# Verify model can be loaded before proceeding
try:
    print('Verifying model access...')
    test_tok = AutoTokenizer.from_pretrained(MODEL_NAME)
    print('✓ Model loaded successfully')
except Exception as e:
    print(f'\n❌ ERROR loading model: {e}')
    if is_kaggle:
        print('\n** On Kaggle, you MUST manually attach the model dataset: **')
        print('  1. Click the "+ Add input" button in the notebook toolbar')
        print('  2. Search for "distilbert-base-uncased" (from datasets)')
        print('  3. Click to add it as input')
        print('  4. Re-run this cell')
    else:
        print('\nFor local use: make sure you have internet access (model will download from HuggingFace)')
    raise

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Build test datasets once for inference
test_ds = Dataset.from_pandas(test_df[['id', 'text']].reset_index(drop=True))
test_ds_swapped = Dataset.from_pandas(
    test_df[['id', 'text_swapped']].rename(columns={'text_swapped': 'text'}).reset_index(drop=True)
)

def tokenize_fn(batch):
    return tokenizer(batch['text'], truncation=True, max_length=MAX_LENGTH)

test_ds = test_ds.map(tokenize_fn, batched=True, remove_columns=['id', 'text'])
test_ds_swapped = test_ds_swapped.map(tokenize_fn, batched=True, remove_columns=['id', 'text'])

data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer,
    pad_to_multiple_of=8 if torch.cuda.is_available() else None
)

print('Prepared fixed test datasets for inference.')

In [ ]:
id2label = {0: 'winner_model_a', 1: 'winner_model_b', 2: 'winner_tie'}
label2id = {v: k for k, v in id2label.items()}

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = torch.softmax(torch.tensor(logits), dim=-1).numpy()
    preds = probs.argmax(axis=1)
    return {
        'accuracy': accuracy_score(labels, preds),
        'log_loss': log_loss(labels, probs, labels=[0, 1, 2])
    }

class WeightedTrainer(Trainer):
    def __init__(self, class_weights=None, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights
        self.model_accepts_loss_kwargs = False

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop('labels')
        outputs = model(**inputs)
        logits = outputs.get('logits')
        if self.class_weights is not None:
            cw = self.class_weights.to(logits.device).to(logits.dtype)
        else:
            cw = None
        loss = F.cross_entropy(logits, labels, weight=cw)
        return (loss, outputs) if return_outputs else loss

def softmax_np(x):
    x = x - np.max(x, axis=1, keepdims=True)
    e = np.exp(x)
    return e / e.sum(axis=1, keepdims=True)

RANDOM_STATE = 42
output_dir = '/kaggle/working/distilbert_preference_model' if Path('/kaggle').exists() else './trained_models/transformer'
use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
use_fp16 = torch.cuda.is_available() and not use_bf16

# ── Train / validation split ──────────────────────────────────────────────────
swap_map = {0: 1, 1: 0, 2: 2}
tr_idx, va_idx = train_test_split(
    np.arange(len(train_df)),
    test_size=0.2,
    stratify=train_df['label'].values,
    random_state=RANDOM_STATE
)

split_train = train_df.iloc[tr_idx][['id', 'text', 'text_swapped', 'label']].copy()
split_valid = train_df.iloc[va_idx][['id', 'text', 'label']].copy()

# Train-time swap augmentation
split_train_swapped = split_train[['id', 'text_swapped', 'label']].rename(columns={'text_swapped': 'text'}).copy()
split_train_swapped['label'] = split_train_swapped['label'].map(swap_map).astype(int)
split_train_aug = pd.concat(
    [split_train[['id', 'text', 'label']], split_train_swapped[['id', 'text', 'label']]],
    ignore_index=True
)

train_ds = Dataset.from_pandas(split_train_aug.reset_index(drop=True))
valid_ds = Dataset.from_pandas(split_valid.reset_index(drop=True))

train_ds = train_ds.map(tokenize_fn, batched=True, remove_columns=['id', 'text'])
valid_ds = valid_ds.map(tokenize_fn, batched=True, remove_columns=['id', 'text'])
train_ds = train_ds.rename_column('label', 'labels')
valid_ds = valid_ds.rename_column('label', 'labels')

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.array([0, 1, 2]),
    y=split_train_aug['label'].values
)
class_weights = torch.tensor(class_weights, dtype=torch.float32)
print('class_weights:', class_weights.tolist())

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3,
    id2label=id2label,
    label2id=label2id
)

# Calculate warmup steps (approximately 10% of total training steps)
total_train_steps = (len(split_train_aug) // (16 * 2)) * 4  # batch_size * grad_accum * epochs
warmup_steps = max(1, int(total_train_steps * 0.1))

training_args = TrainingArguments(
    output_dir=output_dir,
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    gradient_accumulation_steps=2,
    num_train_epochs=4,
    weight_decay=0.01,
    warmup_steps=warmup_steps,
    lr_scheduler_type='cosine',
    eval_strategy='epoch',
    save_strategy='epoch',
    logging_strategy='steps',
    logging_steps=100,
    load_best_model_at_end=True,
    metric_for_best_model='log_loss',
    greater_is_better=False,
    label_smoothing_factor=0.03,
    bf16=use_bf16,
    fp16=use_fp16,
    gradient_checkpointing=True,
    report_to='none',
    save_total_limit=2,
    save_only_model=True,
    seed=RANDOM_STATE
)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=valid_ds,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
    class_weights=class_weights
)
trainer.train()
eval_metrics = trainer.evaluate()
print('Eval metrics:', eval_metrics)

# ── Train prediction features for stacking ─────────────────────────────────────
# Note: predictions on the 80% training split carry leakage;
# predictions on the 20% validation split (va_idx) are held-out.
all_train_ds = Dataset.from_pandas(train_df[['id', 'text']].reset_index(drop=True))
all_train_ds = all_train_ds.map(tokenize_fn, batched=True, remove_columns=['id', 'text'])
train_pred_probs = softmax_np(trainer.predict(all_train_ds).predictions)
val_probs = train_pred_probs[va_idx]
val_logloss = log_loss(train_df['label'].values[va_idx], val_probs, labels=[0, 1, 2])
val_acc = accuracy_score(train_df['label'].values[va_idx], val_probs.argmax(axis=1))
print(f'\nValidation log_loss: {val_logloss:.6f}')
print(f'Validation accuracy: {val_acc:.6f}')
stack_ref_logloss = val_logloss

# ── Test predictions with swap-TTA ────────────────────────────────────────────
base_probs = softmax_np(trainer.predict(test_ds).predictions)
swap_probs = softmax_np(trainer.predict(test_ds_swapped).predictions)
swap_probs_back = swap_probs[:, [1, 0, 2]]
final_probs = (0.5 * base_probs + 0.5 * swap_probs_back).astype(np.float32)
print('Training and inference complete.')

In [ ]:
# ─── Feature Stacking / Meta-Learning ─────────────────────────────────────────
# Blend transformer prediction probabilities with:
#   • Handcrafted signals (length ratios, overlap, readability proxies)
#   • TF-IDF model probabilities (loaded from pre-trained tfidf_chi2_model.joblib)
# Then fit a small LogisticRegression meta-model on the combined prediction features.
# Note: TF-IDF model was trained on the full training set, so its train-set
# predictions carry mild leakage; transformer train predictions include leakage on the training split.

import json as _json
import joblib
from scipy import sparse as _sp
from sklearn.linear_model import LogisticRegression as _LR
from sklearn.preprocessing import StandardScaler as _SS

# ── 1. Raw-text feature helpers (matching the TF-IDF baseline exactly) ────────

def _parse_chat(value):
    if pd.isna(value):
        return ''
    try:
        turns = _json.loads(value)
    except (TypeError, _json.JSONDecodeError):
        return str(value)
    return ' <TURN> '.join(str(t) for t in turns) if isinstance(turns, list) else str(turns)

def _safe_turn_count(value):
    if pd.isna(value):
        return 0
    try:
        turns = _json.loads(value)
    except (TypeError, _json.JSONDecodeError):
        return 1
    return len(turns) if isinstance(turns, list) else 1

def _unique_ratio(text):
    words = str(text).split()
    return len(set(words)) / max(1, len(words))

def _jaccard(text_a, text_b):
    sa = set(str(text_a).lower().split())
    sb = set(str(text_b).lower().split())
    return len(sa & sb) / max(1, len(sa | sb))

def _avg_word_len(text):
    words = str(text).split()
    return float(np.mean([len(w) for w in words])) if words else 0.0

def prepare_meta_features(frame):
    df = frame.copy()
    for col in ['prompt', 'response_a', 'response_b']:
        df[f'{col}_text']  = df[col].map(_parse_chat)
        df[f'{col}_chars'] = df[f'{col}_text'].str.len()
        df[f'{col}_words'] = df[f'{col}_text'].str.split().str.len()
        df[f'{col}_turns'] = df[col].map(_safe_turn_count)

    df['response_chars_diff']    = df['response_a_chars'] - df['response_b_chars']
    df['response_words_diff']    = df['response_a_words'] - df['response_b_words']
    df['response_turns_diff']    = df['response_a_turns'] - df['response_b_turns']
    df['log_resp_chars_ratio']   = np.log1p(df['response_a_chars']) - np.log1p(df['response_b_chars'])
    df['log_resp_words_ratio']   = np.log1p(df['response_a_words']) - np.log1p(df['response_b_words'])
    df['resp_a_unique_ratio']    = df['response_a_text'].map(_unique_ratio)
    df['resp_b_unique_ratio']    = df['response_b_text'].map(_unique_ratio)
    df['unique_ratio_diff']      = df['resp_a_unique_ratio'] - df['resp_b_unique_ratio']

    # Prompt-response word overlap (relevance / helpfulness proxy)
    df['overlap_prompt_a'] = df.apply(
        lambda r: _jaccard(r['prompt_text'], r['response_a_text']), axis=1)
    df['overlap_prompt_b'] = df.apply(
        lambda r: _jaccard(r['prompt_text'], r['response_b_text']), axis=1)
    df['overlap_diff']     = df['overlap_prompt_a'] - df['overlap_prompt_b']

    # Average word length (vocabulary complexity proxy)
    df['avg_word_len_a']    = df['response_a_text'].map(_avg_word_len)
    df['avg_word_len_b']    = df['response_b_text'].map(_avg_word_len)
    df['avg_word_len_diff'] = df['avg_word_len_a'] - df['avg_word_len_b']

    # Columns required by the TF-IDF meta_preprocessor ColumnTransformer
    if 'model_a' not in df.columns:
        df['model_a'] = 'unknown_model_a'
    if 'model_b' not in df.columns:
        df['model_b'] = 'unknown_model_b'
    df['same_model_family'] = df['model_a'].eq(df['model_b'])
    return df

print('Preparing meta-features for train and test...')
train_meta_df = prepare_meta_features(train_df)
test_meta_df  = prepare_meta_features(test_df)

HANDCRAFTED_FEATURES = [
    'prompt_chars', 'prompt_words', 'prompt_turns',
    'response_a_chars', 'response_a_words', 'response_a_turns',
    'response_b_chars', 'response_b_words', 'response_b_turns',
    'response_chars_diff', 'response_words_diff', 'response_turns_diff',
    'log_resp_chars_ratio', 'log_resp_words_ratio',
    'resp_a_unique_ratio', 'resp_b_unique_ratio', 'unique_ratio_diff',
    'overlap_prompt_a', 'overlap_prompt_b', 'overlap_diff',
    'avg_word_len_a', 'avg_word_len_b', 'avg_word_len_diff',
]
print(f'Handcrafted features: {len(HANDCRAFTED_FEATURES)}')

# ── 2. TF-IDF model probabilities ─────────────────────────────────────────────

_tfidf_candidates = [
    'tfidf_chi2_model.joblib',
    '/kaggle/working/tfidf_chi2_model.joblib',
    '/kaggle/input/tfidf-chi2-model/tfidf_chi2_model.joblib',
]
TFIDF_MODEL_PATH = next((p for p in _tfidf_candidates if Path(p).exists()), None)

def _tfidf_predict_proba(meta_df, bundle):
    """Return [N, 3] probabilities aligned to label order [model_a->0, model_b->1, tie->2]."""
    arts  = bundle['artifacts']
    model = bundle['model']
    prompt_block  = arts['prompt_vectorizer'].transform(meta_df['prompt_text'])
    resp_a_block  = arts['response_vectorizer'].transform(meta_df['response_a_text'])
    resp_b_block  = arts['response_vectorizer'].transform(meta_df['response_b_text'])
    diff          = resp_a_block - resp_b_block
    text_block    = _sp.hstack(
        [prompt_block, resp_a_block, resp_b_block, diff.maximum(0), (-diff).maximum(0)],
        format='csr'
    )
    selected      = arts['selector'].transform(text_block)
    meta_block    = arts['meta_preprocessor'].transform(meta_df)
    X             = _sp.hstack([selected, meta_block], format='csr')
    raw_probs     = model.predict_proba(X)
    cls_order     = list(model.classes_)   # ['model_a', 'model_b', 'tie']
    col_idx       = [cls_order.index(c) for c in ['model_a', 'model_b', 'tie']]
    return raw_probs[:, col_idx].astype(np.float32)

if TFIDF_MODEL_PATH:
    print(f'Loading TF-IDF model from: {TFIDF_MODEL_PATH}')
    tfidf_bundle = joblib.load(TFIDF_MODEL_PATH)
    print('Computing TF-IDF predictions on train (mild leakage expected — full-train fit)...')
    tfidf_train_probs = _tfidf_predict_proba(train_meta_df, tfidf_bundle)
    print('Computing TF-IDF predictions on test...')
    tfidf_test_probs  = _tfidf_predict_proba(test_meta_df,  tfidf_bundle)
    print('TF-IDF predictions done.')
else:
    print('WARNING: tfidf_chi2_model.joblib not found — using uniform 1/3 priors for TF-IDF slot.')
    tfidf_train_probs = np.full((len(train_df), 3), 1/3, dtype=np.float32)
    tfidf_test_probs  = np.full((len(test_df),  3), 1/3, dtype=np.float32)

# ── 3. Assemble stacked meta-feature matrix ───────────────────────────────────
# Layout: [transformer_probs(3) | tfidf_probs(3) | handcrafted(23)]

train_handcrafted = train_meta_df[HANDCRAFTED_FEATURES].values.astype(np.float32)
test_handcrafted  = test_meta_df[HANDCRAFTED_FEATURES].values.astype(np.float32)

if 'train_pred_probs' in globals():
    transformer_train_probs = train_pred_probs
elif 'oof_probs' in globals():
    transformer_train_probs = oof_probs
elif 'trainer' in globals() and 'all_train_ds' in globals():
    print('Recomputing transformer train predictions for stacking...')
    transformer_train_probs = softmax_np(trainer.predict(all_train_ds).predictions)
else:
    raise NameError('No transformer train prediction matrix found for stacking.')

meta_X_train = np.hstack([transformer_train_probs, tfidf_train_probs, train_handcrafted])
meta_X_test  = np.hstack([final_probs, tfidf_test_probs,  test_handcrafted])
meta_y_train = train_df['label'].values

print(f'Stacked meta-feature matrix — train: {meta_X_train.shape}, test: {meta_X_test.shape}')

# ── 4. Scale + train LogisticRegression meta-model ────────────────────────────

meta_scaler = _SS()
meta_X_train_sc = meta_scaler.fit_transform(meta_X_train)
meta_X_test_sc  = meta_scaler.transform(meta_X_test)

meta_model = _LR(
    solver='lbfgs',
    C=1.0,
    max_iter=1000,
    random_state=42,
)
meta_model.fit(meta_X_train_sc, meta_y_train)

meta_train_probs = meta_model.predict_proba(meta_X_train_sc)
meta_train_ll    = log_loss(meta_y_train, meta_train_probs, labels=[0, 1, 2])
meta_train_acc   = accuracy_score(meta_y_train, meta_train_probs.argmax(axis=1))
print(f'\n===== Meta-Model (stacked features) =====')
print(f'Train log_loss : {meta_train_ll:.6f}  (reference validation log_loss was: {stack_ref_logloss:.6f})')
print(f'Train accuracy : {meta_train_acc:.6f}')

# Feature importance (mean |coef| across classes)
coef_importance = np.abs(meta_model.coef_).mean(axis=0)
feat_names = (
    ['bert_p_model_a', 'bert_p_model_b', 'bert_p_tie']
    + ['tfidf_p_model_a', 'tfidf_p_model_b', 'tfidf_p_tie']
    + HANDCRAFTED_FEATURES
)
top_feats = sorted(zip(feat_names, coef_importance), key=lambda x: -x[1])[:10]
print('\nTop-10 meta-model features by |coef|:')
for name, score in top_feats:
    print(f'  {score:.4f}  {name}')

# ── 5. Overwrite final_probs with meta-model test predictions ──────────────────
final_probs = meta_model.predict_proba(meta_X_test_sc).astype(np.float32)
print('\nFeature stacking complete — final_probs updated with meta-model output.')


In [ ]:
submission = pd.DataFrame({
    'id': test_df['id'].values,
    'winner_model_a': final_probs[:, 0],
    'winner_model_b': final_probs[:, 1],
    'winner_tie': final_probs[:, 2],
})

submission.to_csv('submission.csv', index=False)
print('Saved submission.csv:', submission.shape)
submission.head()

In [ ]:
print('Model checkpoint saved under:', output_dir)